# FedAvg GĐ2 — train từ đầu, trọng số ngẫu nhiên (Scratch Protocol v2)
Protocol stage2_scratch_clean_v2. Không pretrained ImageNet; không dùng checkpoint PlantVillage cũ.
Mặc định preflight. Actions: preflight, smoke, pilot, run, baseline, collect, flower_verify.
Output phải mới khi fresh. Resume cần RESUME=True và output scratch tương thích.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, os

print("Contents of /kaggle/input:", [p.name for p in Path('/kaggle/input').iterdir()])

# Resolved input paths for Kaggle environment
candidates = list(Path('/kaggle/input').glob('**/stage2_scratch/experiment.py'))
assert len(candidates) == 1, f"Expected exactly 1 package with stage2_scratch/experiment.py, found {len(candidates)}: {candidates}"
PACKAGE_SOURCE = candidates[0].parent.parent
print(f"PACKAGE_SOURCE: {PACKAGE_SOURCE}")

candidate_datasets = [
    Path('/kaggle/input/plantvillage-raw-images/raw/color'),
    Path('/kaggle/input/plantvillage-raw-images/color'),
    *Path('/kaggle/input').glob('**/raw/color'),
    *Path('/kaggle/input').glob('**/color'),
]
valid_datasets = [d for d in candidate_datasets if d.is_dir() and len([p for p in d.iterdir() if p.is_dir()]) == 38]
seen = set()
unique_datasets = [d for d in valid_datasets if not (d in seen or seen.add(d))]
assert len(unique_datasets) == 1, f"Expected exactly 1 valid DATASET_ROOT with 38 classes, found {len(unique_datasets)}: {unique_datasets}"
DATASET_ROOT = unique_datasets[0]
print(f"DATASET_ROOT: {DATASET_ROOT}")

WORK = Path('/kaggle/working/scratch_package')
OUTPUT = Path('/kaggle/working/stage2_scratch_v2')
RESUME = False  # Only True to continue this same scratch experiment
RESUME_SOURCE = None  # Path('/kaggle/input/previous-output/stage2_scratch_v2')
ACTION = 'preflight'  # preflight / smoke / pilot / run / baseline / collect / flower_verify
CONDITIONS = ['label100', 'label1', 'label01']
SEEDS = [42]  # Sau đó 123 và 2026, cùng dữ liệu/cấu hình.
OPTIMIZER = 'sgd'  # sgd (Algorithm 1) hoặc adamw (legacy)
BASELINE_MODE = 'both'  # centralized / local-only / both
SESSION_MINUTES = 420  # Giới hạn phiên; không tự đọc quota tài khoản.

assert (PACKAGE_SOURCE / 'stage2_scratch/experiment.py').is_file()
assert DATASET_ROOT.is_dir()
assert ACTION in {'preflight', 'smoke', 'pilot', 'run', 'baseline', 'collect', 'flower_verify'}


In [ ]:
# Copy only code and frozen manifests. Reused WORK must match uploaded bytes.
folders = ['stage2_scratch', 'stage2_matched', 'stage1_compat', 'src', 'fl_training', 'data/partitions_stage2_scratch_v2']
for folder in folders:
    source, target = PACKAGE_SOURCE / folder, WORK / folder
    assert source.is_dir(), source
    files = [p for p in source.rglob('*') if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc']
    expected = {p.relative_to(source).as_posix() for p in files}
    if target.exists():
        actual = {p.relative_to(target).as_posix() for p in target.rglob('*')
                  if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc'}
        assert actual == expected, 'Stale WORK files: use a new WORK path'
    for p in files:
        dest = target / p.relative_to(source)
        if dest.exists():
            assert dest.read_bytes() == p.read_bytes(), f'Stale code/data: {dest}'
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dest)
if ACTION == 'run':
    if not RESUME:
        if RESUME_SOURCE is not None or OUTPUT.exists():
            raise RuntimeError('Fresh run requires a new OUTPUT and RESUME_SOURCE=None')
    elif RESUME_SOURCE is not None:
        if OUTPUT.exists():
            raise RuntimeError('Restore target must not exist; choose a new OUTPUT')
        if not (RESUME_SOURCE / 'scratch_protocol.json').is_file():
            raise RuntimeError('Only scratch outputs can be restored')
        shutil.copytree(RESUME_SOURCE, OUTPUT)
    elif not (OUTPUT / 'scratch_protocol.json').is_file():
        raise RuntimeError('Resume requires an existing scratch output or RESUME_SOURCE')
os.chdir(WORK)


In [ ]:
# Verify or install compatible PyTorch 2.6.x and torchvision 0.21.x
def check_versions():
    res = subprocess.run(
        [sys.executable, '-c', 'import torch, torchvision; print(torch.__version__.split("+")[0], torchvision.__version__.split("+")[0])'],
        capture_output=True, text=True
    )
    if res.returncode != 0:
        return False, False, f"error: {res.stderr.strip()}"
    parts = res.stdout.strip().split()
    if len(parts) != 2:
        return False, False, f"unexpected: {res.stdout.strip()}"
    t_ver, v_ver = parts
    return t_ver.startswith('2.6.'), v_ver.startswith('0.21.'), f"torch={t_ver}, torchvision={v_ver}"

t_ok, v_ok, current_vers = check_versions()
print(f"Initial environment: {current_vers}")

if not (t_ok and v_ok):
    print("Installing PyTorch 2.6.x / torchvision 0.21.x with matching accelerator build...")
    has_gpu = False
    try:
        res = subprocess.run(['nvidia-smi'], capture_output=True)
        has_gpu = (res.returncode == 0)
    except Exception:
        has_gpu = False
    idx = 'https://download.pytorch.org/whl/cu124' if has_gpu else 'https://download.pytorch.org/whl/cpu'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                    'torch==2.6.0', 'torchvision==0.21.0', '--index-url', idx], check=True)
    t_ok, v_ok, current_vers = check_versions()
    print(f"Post-install environment: {current_vers}")

assert t_ok and v_ok, f"Incompatible versions after bootstrap: {current_vers}"

res = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'],
    capture_output=True, text=True, check=True
)
print("Runtime info:", res.stdout.strip())
if ACTION == 'run':
    assert 'True' in res.stdout.split(), 'Enable GPU accelerator before a full run'

suite = json.loads((WORK / 'data/partitions_stage2_scratch_v2/suite.json').read_text())
assert suite['smoke'] is False and suite['num_clients'] == 5
print({'counts': suite['counts'], 'conditions': CONDITIONS, 'seeds': SEEDS,
       'rounds': 10, 'local_epochs': 1, 'session_minutes': SESSION_MINUTES,
       'pretrained': False, 'initialization': 'random', 'resume': RESUME})

In [ ]:
command = [sys.executable, '-u', '-m', 'stage2_scratch', ACTION,
           '--suite', 'data/partitions_stage2_scratch_v2',
           '--dataset', str(DATASET_ROOT), '--output', str(OUTPUT),
           '--optimizer', OPTIMIZER]
if ACTION == 'run':
    command += ['--conditions', *CONDITIONS, '--seeds', *map(str, SEEDS),
                '--session-minutes', str(SESSION_MINUTES), '--device', 'cuda']
    if RESUME:
        command += ['--resume']
elif ACTION == 'baseline':
    command += ['--mode', BASELINE_MODE, '--conditions', *CONDITIONS, '--seeds', *map(str, SEEDS), '--device', 'cuda']
elif ACTION in ('smoke', 'pilot'):
    command += ['--session-minutes', str(SESSION_MINUTES), '--device', 'cuda']
subprocess.run(command, check=True)


## Tiếp tục và tổng hợp

Để kiểm tra các trục còn lại, đặt CONDITIONS thành
`['quantity100', 'quantity01', 'label_quantity01', 'feature100', 'feature01']`.
Để tiếp tục đúng thí nghiệm scratch, giữ OUTPUT và đặt RESUME=True; để train mới, chọn OUTPUT chưa tồn tại và RESUME=False. Không giảm R/E/batch riêng cho một điều kiện.
Sau các lượt train, đặt ACTION='collect' để đọc `scratch_comparison.json`.
Trước khi phiên kết thúc hãy lưu **toàn bộ OUTPUT** làm artifact.
Collector từ chối data/code/runtime không khớp và không dùng smoke để tính delta.
Luồng này là phần FedAvg; nghiệm thu GĐ2 còn cần đối chứng tập trung/local và fairness matched.